# Week 5 — Gradio demo + HuggingFace Spaces deployment

Builds the same interface as `app.py` (the file your HuggingFace Space will
actually run) so what you test here in Colab is exactly what goes live —
no drive.mount() branch to remember to delete before deploying.


In [ ]:
# Run once per Colab session (safe to re-run: pulls latest instead of
# silently reusing a stale checkout from an earlier run this session).
import os
import subprocess
import sys

REPO_DIR = '/content/repo'
# IMPORTANT: pinned to this specific branch, not the repo's default branch
# -- see Quickstart_Colab.ipynb's Cell 1 for why (an unqualified clone/pull
# can silently track a branch that's missing later fixes, and `git pull`
# then reports "Already up to date" against the wrong branch with no
# warning). Replace with your own fork's URL/branch if running elsewhere,
# or with 'main' once this branch has actually been merged there.
BRANCH = 'claude/high-accuracy-solution-r9r2t1'
if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} --single-branch https://github.com/Rameesha-Malik/ML-SI-Solution-Urdu-OCR.git {REPO_DIR}
else:
    print(f'{REPO_DIR} already exists -- fetching + resetting to origin/{BRANCH} instead of re-cloning.')
    !git -C {REPO_DIR} fetch origin {BRANCH}
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} reset --hard origin/{BRANCH}

os.chdir(REPO_DIR)
!git -C {REPO_DIR} log -1 --format='Repo HEAD: %h %s'

# Deliberately NOT --upgrade-strategy eager, and NOT --force-reinstall on
# the whole file: both would make pip consider reinstalling `torch` even
# though Colab's pre-installed torch already satisfies our ">=2.2.0" bound
# -- and a freshly pip-resolved torch build was observed, in testing, to be
# missing a CUDA sub-package it needs (ImportError: libcublas.so.13), which
# would trade a fixable Python-level bug for a broken GPU runtime. Plain
# `--upgrade -r requirements.txt` still fully resolves and changes
# transformers/pillow/gradio (their pins are strict enough that pip must
# change them regardless of strategy -- upgrade-strategy only affects
# whether *unnamed transitive* dependencies like torch get proactively
# touched), while leaving torch itself alone.
!pip install -q --upgrade -r requirements.txt

# Ground-truth check of what's on disk, via a subprocess (`pip show`) --
# NOT `import transformers`/`import PIL`. If a notebook cell earlier in
# this session already imported one of these, Python caches that module
# in memory for the rest of the process; reinstalling on disk does not
# change what's already loaded there. The actual training/eval scripts
# run as a separate `!python ...` subprocess each time, so they always
# see what's genuinely on disk -- this mirrors that rather than this
# kernel's own (possibly stale) view.
def _installed_version(pkg):
    result = subprocess.run(['pip', 'show', pkg], capture_output=True, text=True)
    return next(
        (l.split(':', 1)[1].strip() for l in result.stdout.splitlines() if l.startswith('Version:')),
        '?',
    )

versions = {pkg: _installed_version(pkg) for pkg in ('transformers', 'pillow', 'gradio')}
print('Installed on disk:', versions)

if 'src' not in sys.path:
    sys.path.insert(0, 'src')

expected_major = {'transformers': ('4.',), 'pillow': ('11.',), 'gradio': ('4.', '5.', '6.0', '6.1')}
bad = {
    p: v for p, v in versions.items()
    if not any(v.startswith(prefix) for prefix in expected_major[p])
}
if bad:
    raise RuntimeError(
        f'Unexpected version(s) on disk: {bad} (expected transformers 4.x, '
        'pillow 11.x, gradio <6.20 per requirements.txt). pip could not '
        'fully satisfy the pins here -- likely another package in this '
        'Colab image forces a conflicting version. Re-run this cell and '
        'share the FULL pip output above (not just this message) so the '
        'real conflicting package can be identified.'
    )
print('Setup complete -- all pinned versions correctly installed on disk.')


## Step 1 — Try the interface in Colab

In [ ]:
import os
# Point at your saved model: a local Drive path while iterating in Colab,
# or your HuggingFace Hub repo id (e.g. 'yourusername/urdu-trocr-si26') once published.
os.environ['MODEL_PATH'] = '/content/drive/MyDrive/SI26-urdu-ocr-model/final'
os.environ['GRADIO_SHARE'] = 'true'

import importlib, app
importlib.reload(app)
app.interface.launch(share=True)


Open the printed `.gradio.live` link, upload one of your Urdu images, and **screenshot the working demo** — you'll need it for your README.

## Step 2 — Deploy on HuggingFace Spaces

1. Go to https://huggingface.co, click your profile icon, **New Space**.
2. Space name: `urdu-ocr-codesaviours-si26-[yourfirstname]`. SDK: **Gradio**. Visibility: **Public**.
3. Push this repo's `app.py`, `requirements.txt`, `src/`, `assets/`, and `examples/` to the Space
   (either `git push` the whole repo to the Space's git remote, or upload the files via the web UI).
4. In the Space's **Settings > Variables and secrets**, add a variable
   `MODEL_PATH` = your HuggingFace Hub model repo id from Week 4
   (e.g. `yourusername/urdu-trocr-si26`) — this is what makes the Space load
   your fine-tuned model instead of failing to find a local `models/` folder.
5. Commit. Wait 2-3 minutes, then visit your Space URL — your app should be live.

Your permanent Space URL:
`https://huggingface.co/spaces/[your-username]/urdu-ocr-codesaviours-si26-[yourfirstname]`

Copy this URL — you need it for your README's **Live demo** section.


## Step 3 — Write your README

Update the main `README.md` (already scaffolded with all 8 required sections —
see the repo root) with:
- your live demo link from Step 2,
- your CER/WER/accuracy numbers and dataset details from Weeks 3-4,
- a screenshot of the working demo,
- your name and credit line.

A strong README takes at least an hour. Don't rush it — it's the single most
visible part of your project.
